# Notes


Model -> Logistic regression

## Download Dataset with the Kaggle API

  Go to https://www.kaggle.com/settings -> API -> Create New Token <br>
  Place the .json file in `C:\Users\yourusername\.kaggle\kaggle.json`<br>

# -

In [1]:
import kagglehub
import os

path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print(f"Dataset downloaded to: {path}")

# Find the CSV file inside the downloaded folder
csv_file = None
for f in os.listdir(path):
    if f.endswith('.csv'):
        csv_file = os.path.join(path, f)
        break

print(f"CSV path: {csv_file}")

c:\Users\BlueberryPancake\miniconda3\envs\DL\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset downloaded to: C:\Users\BlueberryPancake\.cache\kagglehub\datasets\lakshmi25npathi\imdb-dataset-of-50k-movie-reviews\versions\1
CSV path: C:\Users\BlueberryPancake\.cache\kagglehub\datasets\lakshmi25npathi\imdb-dataset-of-50k-movie-reviews\versions\1\IMDB Dataset.csv


In [2]:
# --- Load data ---
import pandas as pd

df = pd.read_csv(csv_file)
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df['sentiment'].value_counts())
df.head(3)

Shape: (50000, 2)

Columns: ['review', 'sentiment']

Label distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive


In [3]:
# --- text preprocessing ---
import re
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # Lowercase
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove special characters and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    tokens = text.split()
    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return ' '.join(tokens)

df['clean_review'] = df['review'].apply(preprocess)

# Show a before/after example
print("Before")
print(df['review'].iloc[0][:300])
print("After")
print(df['clean_review'].iloc[0][:300])

Before
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru
After
one reviewer mentioned watching oz episode youll hooked right exactly happened methe first thing struck oz brutality unflinching scene violence set right word go trust show faint hearted timid show pull punch regard drug sex violence hardcore classic use wordit called oz nickname given oswald maximu


In [4]:
# --- encoding / train test splitting ---
from sklearn.model_selection import train_test_split

# Encode: positive=1, negative=0
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

X = df['clean_review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train)}")
print(f"Test size:  {len(X_test)}")

Train size: 40000
Test size:  10000


In [5]:
# --- TF-IDF vect ---

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print(f"Feature matrix shape: {X_train_tfidf.shape}")

Feature matrix shape: (40000, 20000)


In [7]:
# --- Simple neural net ---

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np


# --- Convert sparse TF-IDF matrices to dense numpy arrays ---
X_train_dense = X_train_tfidf.toarray().astype(np.float32)
X_test_dense  = X_test_tfidf.toarray().astype(np.float32)
y_train_arr   = y_train.values.astype(np.float32)
y_test_arr    = y_test.values.astype(np.float32)


input_dim = X_train_dense.shape[1]

model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# --- Train ---
history = model.fit(
    X_train_dense, y_train_arr,
    epochs=10,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

print("\nTraining complete.")

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 256)               5120256   
                                                                 
 dropout_2 (Dropout)         (None, 256)               0         
                                                                 
 dense_4 (Dense)             (None, 64)                16448     
                                                                 
 dropout_3 (Dropout)         (None, 64)                0         
                                                                 
 dense_5 (Dense)             (None, 1)                 65        
                                                                 
Total params: 5,136,769
Trainable params: 5,136,769
Non-trainable params: 0
_________________________________________________________________
Epoch 1/10
141/141 [========================

In [8]:
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

# --- Run inference ---
probs  = model.predict(X_test_dense, verbose=0).flatten()
y_pred = (probs >= 0.5).astype(int)

accuracy  = accuracy_score(y_test_arr, y_pred)
precision = precision_score(y_test_arr, y_pred)
recall    = recall_score(y_test_arr, y_pred)
f1        = f1_score(y_test_arr, y_pred)


print("BASELINE PERFORMANCE")
print(f"  Accuracy : {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall   : {recall:.4f}")
print(f"  F1-Score : {f1:.4f}")
print()
print(classification_report(y_test_arr, y_pred, target_names=['Negative', 'Positive']))

BASELINE PERFORMANCE
  Accuracy : 0.8779
  Precision: 0.8543
  Recall   : 0.9112
  F1-Score : 0.8818

              precision    recall  f1-score   support

    Negative       0.90      0.84      0.87      5000
    Positive       0.85      0.91      0.88      5000

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000



In [9]:
import joblib

# Save the Keras model (architecture + weights)
model.save('model.h5')

# Save the vectorizer
joblib.dump(vectorizer, 'vectorizer.pkl')

# Save the test split so everyone uses the same split
X_test.to_csv('X_test.csv', index=False)
y_test.to_csv('y_test.csv', index=False)
X_train.to_csv('X_train.csv', index=False)
y_train.to_csv('y_train.csv', index=False)

print("Saved: model.keras, vectorizer.pkl, X_test.csv, y_test.csv")

Saved: model.keras, vectorizer.pkl, X_test.csv, y_test.csv
